# External validation
Benchmarks and boundaries are external inputs. Sources: [LEAD](https://doi.org/10.25984/1784729), [EIA electricity data](https://www.eia.gov/electricity/data.php#sales), and [Census TIGER/Line](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html).

In [ ]:
from pathlib import Path
import os, zipfile
ROOT=Path(os.path.abspath(Path.cwd()))
if ROOT.name=='src':ROOT=ROOT.parent
WORK_DIR=ROOT.parent/'Code_run'
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FormatStrFormatter, MaxNLocator
from shapely.geometry import box
PREDICTIONS_DIR=ROOT.parent/'Predictions'
PREDICTION_PARQUETS=WORK_DIR/'predictions'
OUTPUT_DIR=ROOT/'output'
EXTERNAL_DIR=ROOT / 'external'
VALIDATION_DIR=EXTERNAL_DIR/'validation'
BOUNDARY=EXTERNAL_DIR/'boundaries'
FIG=OUTPUT_DIR/'figures'
plt.rcParams.update({'pdf.fonttype':42,'ps.fonttype':42,'font.family':'DejaVu Sans','font.size':10})
STATE_ABBR=dict(zip([1,4,5,6,8,9,10,12,13,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,44,45,46,47,48,49,50,51,53,54,55,56],
'AL AZ AR CA CO CT DE FL GA ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY'.split()))

In [ ]:
SEASONS=('spring','summer','fall','winter')

def aggregate_predictions(prediction_dir):
    state_parts, annual_parts = [], []
    for season in SEASONS:
        path = Path(prediction_dir) / f"{season}.parquet"
        for batch in pq.ParquetFile(path).iter_batches(batch_size=200_000,
                columns=["GEOID", "YEAR", "MONTH", "PRED_ELEC_M"]):
            frame = batch.to_pandas()
            frame["STATEFIPS"] = frame.GEOID.astype("int64") // 1_000_000_000
            state_parts.append(frame.groupby(["YEAR","MONTH","STATEFIPS"], as_index=False)
                .agg(prediction_sum=("PRED_ELEC_M","sum"), prediction_count=("PRED_ELEC_M","size")))
            annual_parts.append(frame.loc[frame.YEAR.eq(2018), ["GEOID","YEAR","MONTH","PRED_ELEC_M"]])
    state = pd.concat(state_parts).groupby(["YEAR","MONTH","STATEFIPS"], as_index=False)[
        ["prediction_sum","prediction_count"]].sum()
    state["PRED_ELEC_M"] = state.prediction_sum/state.prediction_count
    selected = state[state.STATEFIPS.isin(STATE_ABBR)]
    national = selected.groupby(["YEAR","MONTH"], as_index=False)[["prediction_sum","prediction_count"]].sum()
    national["PRED_ELEC_M"] = national.prediction_sum/national.prediction_count
    monthly = pd.concat(annual_parts, ignore_index=True)
    monthly["GEOID"] = geoid(monthly.GEOID)
    counts = monthly.groupby("GEOID").MONTH.agg(["size","nunique"])
    if not counts["size"].eq(12).all() or not counts["nunique"].eq(12).all():
        raise ValueError("Annual validation requires twelve distinct monthly predictions")
    annual = monthly.groupby(["GEOID","YEAR"], as_index=False).PRED_ELEC_M.sum()
    return state, national, annual.rename(columns={"PRED_ELEC_M":"MONTHLY"})

In [ ]:
def aggregate_release(folder):
    if PREDICTION_PARQUETS is not None:
        state,national,annual=aggregate_predictions(PREDICTION_PARQUETS)
        state=state.rename(columns={'YEAR':'Year','MONTH':'Month','PRED_ELEC_M':'MONTHLY'})
        return state,annual
    state_parts,annual_parts=[],[]
    archives=sorted(Path(folder).glob('*.zip'))
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            members=[n for n in z.namelist() if n.lower().endswith('.csv')]
            for name in sorted(members):
                with z.open(name) as f:d=pd.read_csv(f,usecols=['GEOID','STATEFP','Year','Month','Pred_ELEC'],float_precision='round_trip')
                for col in ['GEOID','STATEFP','Year','Month']:d[col]=pd.to_numeric(d[col],errors='raise').astype('int64')
                state_parts.append(d.groupby(['Year','Month','STATEFP'],as_index=False).agg(prediction_sum=('Pred_ELEC','sum'),prediction_count=('Pred_ELEC','size')))
                annual_parts.append(d.loc[d.Year.eq(2018),['GEOID','Year','Month','Pred_ELEC']])
    state=pd.concat(state_parts,ignore_index=True).rename(columns={'STATEFP':'STATEFIPS'})
    state['MONTHLY']=state.prediction_sum/state.prediction_count
    monthly=pd.concat(annual_parts,ignore_index=True).rename(columns={'Year':'YEAR','Month':'MONTH'})
    monthly['GEOID']=geoid(monthly.GEOID)
    counts=monthly.groupby('GEOID').MONTH.nunique()
    if not counts.eq(12).all():
        raise ValueError('Annual LEAD comparison requires all twelve months.')
    annual=monthly.groupby(['GEOID','YEAR'],as_index=False).Pred_ELEC.sum().rename(columns={'Pred_ELEC':'MONTHLY'})
    return state,annual

In [ ]:
def finite(values, name):
    values = np.asarray(values, dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f"{name}: missing or non-finite values")
    return values

def geoid(values):
    numeric = pd.to_numeric(values, errors="raise")
    numeric = numeric.astype("int64")
    county, tract = numeric // 1_000_000, numeric % 1_000_000
    return county.replace({46113: 46102, 51515: 51019}) * 1_000_000 + tract

def metrics(actual, predicted):
    a, p = finite(actual, "actual"), finite(predicted, "predicted")
    e = p - a
    denominator = np.square(a - a.mean()).sum()
    mask = a != 0
    return dict(n=len(a), r2=float(1 - np.square(e).sum()/denominator) if denominator else None,
                rmse=float(np.sqrt(np.square(e).mean())), mae=float(np.abs(e).mean()),
                mape_percent=float(np.mean(np.abs(e[mask]/a[mask]))*100), bias=float(e.mean()))

def read_eia(folder):
    parts = []
    specs = [("eia_1990_2009.xlsx","Monthly",2009,2009),
             ("eia_2010_onward.xlsx","Monthly-States",2010,2022)]
    for filename, sheet, lo, hi in specs:
        d = pd.read_excel(Path(folder)/filename, sheet_name=sheet, header=None, skiprows=3, usecols=list(range(7)))
        d.columns = ["Year","Month","State","Status","Revenue","Sales","Customers"]
        d["Year"] = pd.to_numeric(d.Year, errors="coerce")
        d["Month"] = pd.to_numeric(d.Month, errors="coerce")
        d = d[d.Year.between(lo,hi) & d.Month.between(1,12) & d.State.isin(STATE_ABBR.values())].copy()
        d[["Year","Month"]] = d[["Year","Month"]].astype(int)
        d["Sales"] = pd.to_numeric(d.Sales, errors="raise")
        d["Customers"] = pd.to_numeric(d.Customers, errors="raise")
        if not d.Customers.gt(0).all():
            raise ValueError("EIA customers must be positive")
        d["EIA_ELEC_M"] = d.Sales*1000/d.Customers
        d["STATEFIPS"] = d.State.map({v:k for k,v in STATE_ABBR.items()})
        parts.append(d[["Year","Month","STATEFIPS","State","EIA_ELEC_M"]])
    result = pd.concat(parts, ignore_index=True)
    return result

def read_lead(folder):
    path = Path(folder)/"lead_2018.dta"
    d = pd.read_stata(path)
    d["GEOID"] = geoid(d.GEOID)
    if "year" in d:
        d = d[pd.to_numeric(d.year).eq(2018)].copy()
    d["YEAR"] = 2018
    d = d[["GEOID","YEAR","LEAD_ELEC_TY"]].copy()
    return d

In [ ]:
def compare_with_benchmarks(predictions_dir,validation_dir):
    state,annual=aggregate_release(predictions_dir)
    lead=read_lead(validation_dir)
    spatial=annual.merge(lead,on=['GEOID','YEAR'],how='inner',validate='one_to_one').dropna(subset=['MONTHLY','LEAD_ELEC_TY'])
    finite(spatial.LEAD_ELEC_TY,'LEAD benchmark')
    eia=read_eia(validation_dir)
    temporal=eia.merge(state[state.STATEFIPS.isin(STATE_ABBR)],on=['Year','Month','STATEFIPS'],how='left',validate='one_to_one')
    finite(temporal.MONTHLY,'state-month prediction')
    report={}
    for name,d,ref in [('LEAD',spatial,'LEAD_ELEC_TY'),('EIA',temporal,'EIA_ELEC_M')]:
        d['APE']=100*(d.MONTHLY-d[ref]).abs()/d[ref]
        report[name]=metrics(d[ref],d.MONTHLY)
    return spatial,temporal,report

spatial,temporal,report=compare_with_benchmarks(PREDICTIONS_DIR,VALIDATION_DIR)
display(pd.DataFrame(report).T[['n','r2','rmse','mape_percent']])

In [ ]:
def save_fig(fig,name):
    FIG.mkdir(parents=True,exist_ok=True)
    for extension in ['png','pdf']:
        fig.savefig(FIG/f'{name}.{extension}',dpi=600,bbox_inches='tight')
    plt.close(fig)

In [ ]:
def basemap(year, projection, clip=True):
    tr = gpd.read_file(BOUNDARY/f'tract_{year}.shp')[['GEOID','geometry']]
    st = gpd.read_file(BOUNDARY/f'state_{year}.shp')[['STATEFP','geometry']]
    tr.GEOID=pd.to_numeric(tr.GEOID).astype('int64')
    tr=tr.to_crs(4326); st=st.to_crs(4326)
    if clip:
        extent=gpd.GeoDataFrame(geometry=[box(-125,24,-66.5,49.5)],crs=4326)
        tr=gpd.overlay(tr,extent,how='intersection')
        st=gpd.overlay(st,extent,how='intersection').dissolve(by='STATEFP')
    else:
        tr=tr[~(tr.GEOID//1000000000).isin([2,15,60,66,69,72,78])]
        st=st[~pd.to_numeric(st.STATEFP).isin([2,15,60,66,69,72,78])]
    return tr.to_crs(projection),st.to_crs(projection)

def scatter(ax,x,y,met,labx,laby,equal=False,pretty=False):
    x=np.asarray(x,float);y=np.asarray(y,float)
    hist,xe,ye=np.histogram2d(x,y,bins=70)
    ix=np.clip(np.searchsorted(xe,x,side='right')-1,0,69);iy=np.clip(np.searchsorted(ye,y,side='right')-1,0,69)
    density=np.log1p(hist[ix,iy])
    order=np.argsort(density)
    ax.scatter(x[order],y[order],c=density[order],cmap='viridis',s=10,alpha=.85,linewidths=0)
    lo=min(x.min(),y.min());hi=max(x.max(),y.max());pad=(hi-lo)*.03;lo=max(0,lo-pad);hi+=pad
    ax.plot([lo,hi],[lo,hi],color='black',lw=1,alpha=.8)
    ax.set_xlim(lo,hi);ax.set_ylim(lo,hi)
    if equal: ax.set_aspect('equal',adjustable='box')
    if pretty:
        raw=(hi-lo)/4; mag=10**np.floor(np.log10(raw)); candidates=np.array([1,2,2.5,5,10])*mag
        step=candidates[candidates >= raw][0]
        ticks=np.arange(np.floor(lo/step)*step,np.ceil(hi/step)*step+step*.1,step)
        ax.set_xticks(ticks);ax.set_yticks(ticks)
        ax.set_xlim(lo,hi);ax.set_ylim(lo,hi)
    else:
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'));ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    for s in ax.spines.values():s.set_linewidth(1.5)
    ax.set_xlabel(labx,fontsize=12);ax.set_ylabel(laby,fontsize=12);ax.tick_params(labelsize=10)
    label=f"$R^2$ = {met['r2']:.2f}\nRMSE = {met['rmse']:.2f}\nMAPE = {met['mape_percent']:.2f}%"
    ax.text(.02,.98,label,transform=ax.transAxes,va='top',ha='left',fontsize=12,bbox={'boxstyle':'round,pad=0.25','facecolor':'white','alpha':.85,'edgecolor':'none'})

def figure6(spatial,report):
    tr,st=basemap(2019,2163,clip=False)
    tr=tr.merge(spatial[['GEOID','APE']],on='GEOID',how='left',validate='many_to_one')
    cap=float(tr.APE.quantile(.99));tr['APE_display']=tr.APE.clip(upper=cap)
    fig=plt.figure(figsize=(10.6,4.8),constrained_layout=True)
    gs=fig.add_gridspec(1,2,width_ratios=[1.2,1],wspace=.08)
    left=gs[0].subgridspec(2,1,height_ratios=[20,.8],hspace=.02)
    axmap=fig.add_subplot(left[0]);cax=fig.add_subplot(left[1]);axs=fig.add_subplot(gs[1])
    tr.plot(column='APE_display',ax=axmap,cmap='viridis',vmin=0,vmax=cap,linewidth=0,edgecolor='none',missing_kwds={'color':'lightgrey'},rasterized=True)
    st.boundary.plot(ax=axmap,color='black',linewidth=.5)
    axmap.set_axis_off()
    cb=fig.colorbar(ScalarMappable(norm=Normalize(0,cap),cmap='viridis'),cax=cax,orientation='horizontal',format=FormatStrFormatter('%.2f'))
    cb.set_label('Absolute Percentage Error (%)',fontsize=12,labelpad=3);cb.ax.tick_params(labelsize=10);cb.outline.set_linewidth(.8)
    scatter(axs,spatial.LEAD_ELEC_TY,spatial.MONTHLY,report['LEAD'],'LEAD Annual Electricity Consumption (kWh)','Predicted Annual Electricity Consumption (kWh)',equal=True,pretty=True)
    fig.canvas.draw()
    pmap,pscatter=axmap.get_position(),axs.get_position()
    label_y=max(pmap.y1,pscatter.y1)+.025
    fig.text(pmap.x0-.035,label_y,'a',fontsize=16,fontweight='bold')
    fig.text(pscatter.x0-.035,label_y,'b',fontsize=16,fontweight='bold')
    save_fig(fig,'Figure6')

def figure7(temporal,report):
    monthly=temporal.groupby(['Year','Month'],as_index=False).APE.mean()
    monthly['date']=pd.to_datetime({'year':monthly.Year,'month':monthly.Month,'day':1})
    fig,axs=plt.subplots(1,2,figsize=(12.5,4.8),constrained_layout=True,gridspec_kw={'width_ratios':[1.15,1]})
    ax=axs[0];ax.plot(monthly.date,monthly.APE,color='black',lw=1.4)
    ax.set_xlim(pd.Timestamp('2009-01-01'),pd.Timestamp('2022-12-01'))
    ax.set_ylabel('Mean Absolute Percentage Error (%)',fontsize=12);ax.set_xlabel('')
    ax.xaxis.set_major_locator(mdates.YearLocator(2));ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'));ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'));ax.tick_params(labelsize=10)
    for s in ax.spines.values():s.set_linewidth(1.5)
    scatter(axs[1],temporal.EIA_ELEC_M,temporal.MONTHLY,report['EIA'],'EIA Monthly Electricity Consumption (kWh)','Predicted Monthly Electricity Consumption (kWh)')
    for letter,ax in zip('ab',axs): ax.text(-.10,1.05,letter,transform=ax.transAxes,fontsize=18,fontweight='bold',clip_on=False)
    save_fig(fig,'Figure7')

In [ ]:
figure6(spatial,report)
figure7(temporal,report)